In [ ]:
from unstructured.partition.pdf import partition_pdf
from IPython.display import Image, display
import base64
import pandas as pd
import io
import tabulate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv
from collections import defaultdict
import json
import psycopg2
from pgvector.psycopg2 import register_vector


In [15]:
load_dotenv()

True

In [16]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [5]:
chunks = partition_pdf(
    filename="government-data-security-policies.pdf",
    strategy="hi_res",

    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=True,

    chunking_strategy="by_title",
    max_characters=2000,
    combine_text_under_n_chars=1000,
    new_after_n_chars=1500
)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3203.77it/s]


In [ ]:
string = chunks[20].metadata.orig_elements[0].metadata.image_base64
display(Image(data=base64.b64decode(string)))

In [6]:
def modify_text(chunk):
    text = []
    for element in chunk.metadata.orig_elements:
        if element.category == "Table" and getattr(element.metadata, "text_as_html", None):
            df = pd.read_html(io.StringIO(element.metadata.text_as_html))[0]
            markdown = df.to_markdown(index=False)
            text.append(markdown)
        else:
            text.append(element.text)
    chunk.text = "\n\n".join(text)

In [7]:
def extract_coords(chunk):
    highlights = []
    orig_elements = chunk.metadata.orig_elements
    for element in orig_elements:
        page = element.metadata.page_number
        coords = element.metadata.coordinates.points

        width = element.metadata.coordinates.system.width
        height = element.metadata.coordinates.system.height

        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
    
        clean_box = [
            round(max(0, float(min(xs))),2),
            round(max(0, float(min(ys))),2),
            round(min(width, float(max(xs))), 2),
            round(min(height, float(max(ys))), 2)
        ]
        highlights.append({
            "page": int(page),
            "bbox": clean_box,
            "width": int(width),
            "height": int(height)
        })
    setattr(chunk.metadata, "highlights", highlights)

In [17]:
def embed_text(chunk):
    vector = embeddings.embed_query(chunk.text)
    setattr(chunk.metadata, "vector", vector)

In [18]:
def process_chunk(chunk):
    modify_text(chunk)
    embed_text(chunk)
    extract_coords(chunk)

In [19]:
for chunk in chunks:
    process_chunk(chunk)

GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}

In [10]:
chunks[0].metadata.highlights

[{'page': 1, 'bbox': [0, 0, 2894, 4093], 'width': 2894, 'height': 4093},
 {'page': 1,
  'bbox': [840.05, 707.56, 2728.42, 1500.48],
  'width': 2894,
  'height': 4093},
 {'page': 1,
  'bbox': [1263.15, 1791.99, 2614.9, 2504.62],
  'width': 2894,
  'height': 4093},
 {'page': 1,
  'bbox': [2432.32, 2448.24, 2663.55, 2506.58],
  'width': 2894,
  'height': 4093},
 {'page': 1,
  'bbox': [1787.9, 3836.39, 2618.12, 3885.83],
  'width': 2894,
  'height': 4093},
 {'page': 2, 'bbox': [0, 0, 2894, 231.0], 'width': 2894, 'height': 4093}]

In [11]:
import fitz
def visualize_chunk_highlights(pdf_path, chunk, output_path="preview.pdf"):
    doc = fitz.open(pdf_path)
    highlights = getattr(chunk.metadata, "highlights", [])

    for h in highlights:
        page_index = h['page'] - 1
        page = doc[page_index]
        
        # 1. Get the page size as PyMuPDF sees it (72 DPI)
        actual_w = page.rect.width
        actual_h = page.rect.height
        
        # 2. Get the original system size from your metadata
        orig_w = h['width']
        orig_h = h['height']
        
        # 3. SCALE the coordinates
        # Math: (Raw_Value / Original_System_Size) * Actual_PDF_Size
        raw_bbox = h['bbox'] # [x1, y1, x2, y2]
        
        scaled_bbox = [
            (raw_bbox[0] / orig_w) * actual_w, # x_min
            (raw_bbox[1] / orig_h) * actual_h, # y_min
            (raw_bbox[2] / orig_w) * actual_w, # x_max
            (raw_bbox[3] / orig_h) * actual_h  # y_max
        ]
        
        # 4. Draw the correctly scaled rectangle
        rect = fitz.Rect(scaled_bbox)
        page.draw_rect(rect, color=(1, 1, 0), fill=(1, 1, 0), overlay=True, width=0)

    doc.save(output_path)
visualize_chunk_highlights("government-data-security-policies.pdf", chunks[4], output_path="preview.pdf")

In [168]:
chunks[3].text

'Section 2: Technical and process measures to protect data and prevent data compromises\n\nAgencies should implement the appropriate technical and process measures to protect 05 data against data security threats and prevent compromises of the conﬁdentiality and integrity of the data.\n\nAgencies should adopt a risk-based approach when implementing the technical and process measures. Technical measures are only eﬀective when the complementary process measures are implemented too (See Annex A).\n\nThe data security technical and process measures are to be implemented on top of any cybersecurity measures in place.\n\nData should be protected according to the risk level determined through a data security risk assessment. Agencies should ensure that data is adequately protected while minimising the impact to operations, resources and costs.\n\n06 Agencies should adopt the following strategies when implementing the technical measures and process measures to safeguard data:\n\na) Reduce the 

In [14]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()